In [1]:
%reload_kedro

                    INFO     Resolved project path as:                                              ]8;id=196632;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=511228;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py#175\175]8;;\
                             C:\Users\nicolas.betancourt\Documents\GitHub\pytorch\serpientes-de-col                
                             ombia.                                                                                
                             To set a different path, run '%reload_kedro <project_root>'                           

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=208798;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=745461;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro_telemetry\plugin.py#233\233]8;;\
                             the product. No personal data or IP addresses are stored on our side. If              
                             you want to opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK`              
                             environment variables, or create a `.telemetry` file in the current                   
                             working directory with the contents `consent: false`. Read more at                    
                             https://docs.kedro.org/en/stable/configuration/telemetry.html                         

[07/29/25 08:14:31] INFO     Kedro project serpientes_de_colombia                                   ]8;id=848750;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=292876;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py#141\141]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=846919;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=27247;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py#142\142]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=194609;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=187737;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py#148\148]8;;\

In [2]:
import torch
from torch.utils.data import Dataset,  DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import torchvision
from torchvision import datasets, models, transforms
from torchvision.transforms import ToTensor, Lambda



import os
import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from PIL import Image
from tempfile import TemporaryDirectory
matplotlib.use('Agg')
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
train_image_dataset=catalog.load("train_image_dataset" )
test_image_dataset =catalog.load("test_image_dataset"  )
training_params    =catalog.load("params:dense_params"  )
label_map          =catalog.load('label_encoder'       )

[07/29/25 08:14:32] INFO     Loading data from train_image_dataset                              ]8;id=556839;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=994633;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py#539\539]8;;\
                             (KedroPytorchImageDataset)...                                                         

                    INFO     Loading data from test_image_dataset (KedroPytorchImageDataset)... ]8;id=977185;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=682250;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py#539\539]8;;\

                    INFO     Loading data from params:dense_params (MemoryDataset)...           ]8;id=976460;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=151738;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py#539\539]8;;\

                    INFO     Loading data from label_encoder (JSONDataset)...                   ]8;id=487936;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=47164;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py#539\539]8;;\

In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


transform=transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
target_transform=lambda x: label_map.get(x)
training_set = train_image_dataset.with_transforms(transform=transform, target_transform=target_transform)
training_generator = DataLoader(training_set, **training_params)

Using cuda device


In [6]:
model_conv = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in model_conv.parameters():
    param.requires_grad = False

# Parameters of newly constructed modules have requires_grad=True by default
num_ftrs = model_conv.fc.in_features
num_classes=train_image_dataset.data[train_image_dataset.label_column].nunique()
model_conv.fc = nn.Linear(num_ftrs, num_classes)

model_conv = model_conv.to(device)

criterion = nn.CrossEntropyLoss()


optimizer_conv = optim.SGD(model_conv.fc.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_conv, step_size=7, gamma=0.1)

# Función auxiliar 
El proceso con todos los modelos es el mismo, entonces es buena cosa tener una función auxiliar que reciba:

* Modelo
* Datasets
* Criterio de optimización (métrica)
* El optimizador
* Tal vez un scheduler para el lr
* Num epochs
* Si validación o si train

Y retorne el modelo ajustado. Eso lo hacemos acá.

In [20]:
def model_calibration(model, dataloader, criterion, optimizer, scheduler, phase='train',num_epochs=25):
    best_accuracy = 0.0
    since = time.time()
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)
    
        # Each epoch has a training and validation phase
        #for phase in ['train', 'val']:
        if phase == 'train':
            model.train()  # Set model to training mode
        else:
            model.eval()   # Set model to evaluate mode
    
        running_loss = 0.0
        running_corrects = 0
    
        # Iterate over data.
        for inputs, labels, path in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
    
            # zero the parameter gradients
            optimizer.zero_grad()
    
            # forward
            # track history if only in train
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
    
                # backward + optimize only if in training phase
                if phase == 'train':
                    loss.backward()
                    optimizer.step()
    
            # statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
        if phase == 'train':
            scheduler.step()
    
            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_accuracy = running_corrects.double() / len(dataloader.dataset)
    
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_accuracy:.4f}')
    
            # deep copy the model
            if phase == 'val' and epoch_accuracy > best_accuracy:
                best_accuracy = epoch_accuracy
                
    
       
    
    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    if phase == 'val':
        print(f'Best val Acc: {best_accuracy:4f}')
    
    # load best model weights
    return model

# Acá miramos si la función sirvió o no 

In [23]:
model_conv=model_calibration(model_conv, training_generator, criterion, optimizer_conv, exp_lr_scheduler, phase='train',num_epochs=25)

Epoch 0/24
----------
train Loss: 0.1864 Acc: 0.9401
Epoch 1/24
----------
train Loss: 0.2193 Acc: 0.9085
Epoch 2/24
----------
train Loss: 0.1904 Acc: 0.9369
Epoch 3/24
----------
train Loss: 0.2456 Acc: 0.8896
Epoch 4/24
----------
train Loss: 0.2156 Acc: 0.9022
Epoch 5/24
----------
train Loss: 0.1709 Acc: 0.9401
Epoch 6/24
----------
train Loss: 0.1994 Acc: 0.9274
Epoch 7/24
----------
train Loss: 0.1852 Acc: 0.9274
Epoch 8/24
----------
train Loss: 0.1811 Acc: 0.9401
Epoch 9/24
----------
train Loss: 0.1952 Acc: 0.9243
Epoch 10/24
----------
train Loss: 0.1997 Acc: 0.9338
Epoch 11/24
----------
train Loss: 0.2710 Acc: 0.8959
Epoch 12/24
----------
train Loss: 0.2370 Acc: 0.9022
Epoch 13/24
----------
train Loss: 0.2205 Acc: 0.9148
Epoch 14/24
----------
train Loss: 0.2094 Acc: 0.9243
Epoch 15/24
----------
train Loss: 0.2372 Acc: 0.9117
Epoch 16/24
----------
train Loss: 0.2116 Acc: 0.9148
Epoch 17/24
----------
train Loss: 0.1907 Acc: 0.9243
Epoch 18/24
----------
train Loss: 0.1